In [1]:
import os
import numpy as np
import pandas as pd

from dataclasses import field
from pydantic import BaseModel, Field,  field_validator
from typing import List, Tuple, Literal
from openai import OpenAI

import SoD_Utils
import Text_Utils

In [2]:
LANGUAGE = SoD_Utils.LANGUAGE_CZ

data_path = path = SoD_Utils.get_dataset_path(LANGUAGE)
df = pd.read_spss(data_path)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3880 entries, 0 to 3879
Columns: 170 entries, id to lca_DK_n_8
dtypes: category(163), float64(7)
memory usage: 864.0 KB


In [16]:
def process_SoD_response(row):
    # --- 1. Basic Information ---
    # Renames Czech keys to English and performs initial data cleaning.
    processed_data = {
        'gender': row.get('GENDER').lower(),
        'age': int(row.get('AGE1', 0)),
        'education_level': row.get('EDU', '').lower(),
        'region': row.get('KRAJ'),
        'district': row.get('OKRES'),
        'town_size': SoD_Utils.process_town_size(row),
        'employment_status': SoD_Utils.process_employment(row),
        'income_range': SoD_Utils.process_income(row),
        'voted_party': row.get('Q21')
    }

    # --- 2. Additional Survey Questions ---
    # Merges the dictionary of additional questions into the main one.
    additional_data = {
        'living_standard': row.get('Q19','').lower(),
        'interest_in_politics': row.get('Q20','').lower(),
        'opinion_on_eu': row.get('Q18','').lower(),
        'opinion_on_nato': row.get('Q17','').lower(),
        'covid_vaccinated': row.get('Q23','').lower(),
    }
    processed_data.update(additional_data)


    return processed_data

data = [process_SoD_response(df.loc[i]) for i in df.index]
data = pd.DataFrame(data, index=df.index)

In [4]:
data

,gender,age,education_level,region,district,town_size,employment_status,income_range,living_standard,interest_in_politics,opinion_on_eu,opinion_on_nato,covid_vaccinated
0,muž,28,vysokoškolské vzdělání,Středočeský kraj,Nymburk,Méně než 1.000 obyvatel,zaměstnanec na plný úvazek,30.001 - 40.000 Kč,spíše dobrou,velmi se zajímám,spíše spokojený/á,rozhodně spokojený/á,ano
1,muž,27,vysokoškolské vzdělání,Plzeňský kraj,Plzeň-město,Více než 100.000 obyvatel,zaměstnanec na plný úvazek,20.001 - 25.000 Kč,spíše dobrou,spíše se zajímám,spíše spokojený/á,spíše spokojený/á,ano
2,žena,33,základní + středoškolské vzdělání bez maturity,Královéhradecký kraj,Jičín,Méně než 1.000 obyvatel,zaměstnanec na plný úvazek,25.001 - 30.000 Kč,"ani dobrou, ani špatnou",spíše se zajímám,spíše spokojený/á,rozhodně spokojený/á,ano
3,žena,27,základní + středoškolské vzdělání bez maturity,Plzeňský kraj,Rokycany,20.001 - 100.000 obyvatel,zaměstnanec na plný úvazek,20.001 - 25.000 Kč,spíše dobrou,spíše se zajímám,rozhodně spokojený/á,rozhodně spokojený/á,ano
4,muž,21,středoškolské vzdělání s maturitou,Ústecký kraj,Chomutov,20.001 - 100.000 obyvatel,zaměstnanec na plný úvazek,30.001 - 40.000 Kč,velmi dobrou,vůbec se nezajímám,rozhodně spokojený/á,rozhodně spokojený/á,ano
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3875,žena,39,středoškolské vzdělání s maturitou,Plzeňský kraj,Plzeň-město,Více než 100.000 obyvatel,zaměstnanec na plný úvazek,40.001 - 60.000 Kč,spíše dobrou,spíše se nezajímám,spíše spokojený/á,spíše spokojený/á,ne
3876,žena,30,vysokoškolské vzdělání,Hlavní město Praha,Praha,Více než 100.000 obyvatel,zaměstnanec na částečný úvazek,None,spíše dobrou,spíše se zajímám,rozhodně spokojený/á,rozhodně spokojený/á,ano
3877,žena,47,vysokoškolské vzdělání,Moravskoslezský kraj,Karviná,5.001 - 20.000 obyvatel,zaměstnanec na plný úvazek,40.001 - 60.000 Kč,spíše dobrou,vůbec se nezajímám,spíše spokojený/á,rozhodně spokojený/á,ano
3878,žena,23,středoškolské vzdělání s maturitou,Hlavní město Praha,Praha,20.001 - 100.000 obyvatel,zaměstnanec na plný úvazek,20.001 - 25.000 Kč,spíše špatnou,nevím,nevím,nevím,nechci uvést


In [5]:
def _format_opinion_statement(gender, opinion, topic_string):
    """
    Creates a full opinion sentence with correct gendered adjectives.
    Example: (gender='Muž', opinion='Rozhodně ano', topic='EU')
             -> "Jsem rozhodně přesvědčený, že je Česká republika členem EU."

    This helper removes code duplication for the EU and NATO questions.
    """
    if opinion == 'Nevím':
        return "" # Return an empty string if there is no opinion

    return f"Jsem {Text_Utils.declension_gender_ya(opinion[:-3],gender).lower()}, že je Česká republika členským státem {topic_string}."
# --- Main Function to Create the Description ---

def create_respondent_description(respondent):
    """
    Generates a descriptive Czech paragraph about a survey respondent
    by combining their answers into grammatically correct sentences.

    Args:
        respondent (dict): A dictionary containing the processed data for one person,
                           with English keys (e.g., 'gender', 'region').

    Returns:
        str: A multi-sentence description of the respondent in Czech.
    """
    # --- 1. Build the description sentence by sentence ---
    # Using a list of sentences is cleaner than repeated string concatenation.
    gender = SoD_Utils.gender_to_enum_gender(respondent['gender'])
    description_parts = []

    # --- Basic Demographics ---
    description_parts.append(
        f"Jsem {respondent['gender']}, "
        f"je mi {respondent['age']} let, "
        f"mé vzdělání je {respondent['education_level']}."
    )

    # --- Location ---
    # This now uses the helper function for complex Czech grammar.
    description_parts.append(
        f"Žiji v {SoD_Utils.decline_region_to_Locative(respondent['region'])}, "
        f"v okresu {respondent['district']} a "
        f"obci o velikosti {respondent['town_size']}."
    )

    # --- Socioeconomic Status ---
    description_parts.append(f"Z hlediska zaměstnání jsem {respondent['employment_status']}")
    if respondent['income_range']:
        description_parts.append(f"a příjem naší domácnosti je {respondent['income_range']}")
    description_parts[-1]+="."

    # EU and NATO opinions now use the dedicated helper function
    description_parts.append(_format_opinion_statement(gender, respondent['opinion_on_eu'], "EU"))
    description_parts.append(_format_opinion_statement(gender, respondent['opinion_on_nato'], "NATO"))

    # Living Standard
    living_standard = respondent['living_standard']
    if SoD_Utils.is_non_substantive_responses(living_standard):
        verb = 'mám'
        if 'ani' in respondent['living_standard']:
            verb = Text_Utils.declension_negation_ne(verb)
        description_parts.append(f"{verb.capitalize()} {living_standard} životní úroveň.")

    # Interest in Politics
    interest = respondent['interest_in_politics']
    if SoD_Utils.is_non_substantive_responses(interest):
        description_parts.append(f"{interest.capitalize()} o politiku.")

    # COVID Vaccination Status
    vacc_status = respondent['covid_vaccinated']
    if SoD_Utils.is_non_substantive_responses(vacc_status):
        is_vaccinated = Text_Utils.parse_yes_no(vacc_status)
        verb = 'jsem'
        if not is_vaccinated:
            verb = Text_Utils.declension_negation_ne(verb)
        description_parts.append(f"{verb.capitalize()} {Text_Utils.declension_gender_ya('očkován',gender)} proti covidu.")

    # --- 2. Combine all parts into a final paragraph ---
    # Filter out any empty strings that may have been returned by helpers (e.g., for 'Nevím' answers)
    # and join the parts with a space.
    full_description = " ".join(part for part in description_parts if part)

    return full_description

In [6]:
create_respondent_description(data.iloc[1687])

'Jsem žena, je mi 42 let, mé vzdělání je středoškolské vzdělání s maturitou. Žiji v Ústeckém kraji, v okresu Chomutov a obci o velikosti 5.001 - 20.000 obyvatel. Z hlediska zaměstnání jsem zaměstnanec na plný úvazek. Jsem neá, že je Česká republika členským státem EU. Jsem neá, že je Česká republika členským státem NATO.'

## Calling prompts

In [7]:
INSTRUCTIONS = """
Jsi analytik volebního chování v ČR. Na základě profilu respondenta odhadni:

voted_or_not: pravděpodobnost, že ve volbách do PS 2021 volil vs. nevolil (součet = 1.0).
parties: rozdělení P(strana | volil) pouze mezi povolené položky schématu (součet = 1.0). Používej přesné názvy stran ze schématu. Pokud žádná konkrétní strana výrazně nevyčnívá, použij „Jiná strana“. Neuváděj žádný volný text. Pokud jsou v profilu placeholdery jako [INSERT], považuj je za neznámé. Kontekst: ČR, sněmovní volby 2021. Neuváděj neexistující subjekty ani fakta. """

INSTRUCTIONS2 ="""
Jsi expertní AI asistent specializovaný na analýzu českého politického chování. Tvým úkolem je na základě demografického a postojového profilu respondenta odhadnout jeho volební chování ve volbách do Poslanecké sněmovny Parlamentu ČR v roce 2021.

Tvůj výstup MUSÍ být JSON objekt, který přesně odpovídá Pydantic modelu `VotingResult`. Jiný formát není přípustný.

Dodržuj tato pravidla:
1.  Analyzuj VŠECHNY poskytnuté informace o respondentovi (věk, vzdělání, bydliště, příjem, postoje k EU/NATO atd.).
2.  Na základě analýzy odhadni dvě klíčové věci:
    a) Jaká je pravděpodobnost, že respondent vůbec šel k volbám.
    b) Pokud volil, jaké jsou pravděpodobnosti pro jednotlivé politické strany.
3.  Vygeneruj JSON, který bude validní oproti poskytnutým Pydantic modelům.
4.  Dbej na to, aby součet pravděpodobností v objektu `voted_or_not` byl přesně 1.0.
5.  Dbej na to, aby součet pravděpodobností VŠECH stran v seznamu `parties` byl přesně 1.0."""


czech_prompt_end = " Ve volbách do poslanecké sněmovny 2021 jsem [INSERT]"

In [8]:
#TODO: GPT 4 nano
# pydantic structured data DONE?
# poslat vysledky
# ciel : structured output works

In [9]:
class PartyProbability(BaseModel):
    """
    Strukturovaná reprezentace pravděpodobnosti hlasování pro konkrétní stranu.
    """
    name: Literal[
        "ANO 2011",
        "Koalice Spolu (ODS, TOP 09, KDU-ČSL)",
        "Koalice PIRÁTI a STAROSTOVÉ",
        "Komunistická strana Čech a Moravy (KSČM)",
        "Svoboda a přímá demokracie – Tomio Okamura (SPD)",
        "Česká strana sociálně demokratická (ČSSD)",
        "Trikolóra, Svobodní a Soukromníci",
        "Přísaha Roberta Šlachty",
        "Jiná strana"
    ] = Field(description="Název politické strany.")
    probability: float = Field(
        ge=0, le=1, description="Pravděpodobnost, že respondent hlasoval pro tuto stranu."
    )

class VotedProbability(BaseModel):
    """
    Pravděpodobnost, zda respondent volil, nebo nevolil.
    """
    voted: float = Field(ge=0, le=1, description="Pravděpodobnost, že respondent volil.")
    not_voted: float = Field(ge=0, le=1, description="Pravděpodobnost, že respondent nevolil.")

    # @field_validator('*', pre=True, always=True)
    # def check_sum(cls, v, values):
    #     # Tento validátor zajistí, že součet se bude blížit 1.0
    #     # Můžete si pohrát s tolerancí (např. 0.01)
    #     if 'voted' in values and 'not_voted' in values:
    #         if abs(values['voted'] + values['not_voted'] - 1.0) > 0.001:
    #             raise ValueError("Součet pravděpodobností pro 'voted' a 'not_voted' se musí rovnat 1.0")
    #     return v

class VotingResult(BaseModel):
    """
    Strukturovaný výstup pro volební chování respondenta ve volbách do poslanecké sněmovny 2021.
    """
    # Změna zde: použití nové třídy místo složitého Tuple
    voted_or_not: VotedProbability = Field(
        description="Pravděpodobnost, zda respondent volil, nebo nevolil. Součet musí být 1.0."
    )
    parties: List[PartyProbability] = Field(
        description="Seznam možných stran s pravděpodobností volby. Součet pravděpodobností musí být 1.0."
    )

In [18]:
respondent = data.iloc[0]
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
input = create_respondent_description(respondent) + czech_prompt_end

In [11]:
response = client.responses.parse(
    model="gpt-4.1-nano",
    #instructions="You are a summarizer that returns 'result' summary and a 'confidence' score.",
    input=input,
    text_format=VotingResult
)

response.output_parsed

VotingResult(voted_or_not=VotedProbability(voted=0.6, not_voted=0.4), parties=[PartyProbability(name='ANO 2011', probability=0.15), PartyProbability(name='Koalice Spolu (ODS, TOP 09, KDU-ČSL)', probability=0.25), PartyProbability(name='Koalice PIRÁTI a STAROSTOVÉ', probability=0.2), PartyProbability(name='Komunistická strana Čech a Moravy (KSČM)', probability=0.05), PartyProbability(name='Svoboda a přímá demokracie – Tomio Okamura (SPD)', probability=0.15), PartyProbability(name='Česká strana sociálně demokratická (ČSSD)', probability=0.1), PartyProbability(name='Trikolóra, Svobodní a Soukromníci', probability=0.05), PartyProbability(name='Přísaha Roberta Šlachty', probability=0.02), PartyProbability(name='Jiná strana', probability=0.03)])

In [19]:
voted = response.output_parsed

p = 0
for party_vote in voted.parties:
    print(f"{party_vote.name}: {party_vote.probability}")
    p += party_vote.probability
print(p)

respondent["voted_party"]

ANO 2011: 0.15
Koalice Spolu (ODS, TOP 09, KDU-ČSL): 0.25
Koalice PIRÁTI a STAROSTOVÉ: 0.2
Komunistická strana Čech a Moravy (KSČM): 0.05
Svoboda a přímá demokracie – Tomio Okamura (SPD): 0.15
Česká strana sociálně demokratická (ČSSD): 0.1
Trikolóra, Svobodní a Soukromníci: 0.05
Přísaha Roberta Šlachty: 0.02
Jiná strana: 0.03
1.0000000000000002


'Koalice PIRÁTI a STAROSTOVÉ'